# AML ranking comparison

Does prior account activity help a simulated analyst find more transactions labelled as laundering within the same review capacity?

This notebook starts with the MySQL export for 2–8 September 2022 in IBM's synthetic data. The 9–10 September test labels are reserved for the frozen comparison.

In [1]:
# Step 22. Load and verify the development export.
from pathlib import Path

import pandas as pd
from IPython.display import display

columns = [
    "source_row_id",
    "transaction_timestamp",
    "split_name",
    "amount_paid",
    "payment_currency",
    "prior_24h_count",
    "is_laundering",
]

development = pd.read_csv(
    Path("data/model/aml_ranking_development.tsv"),
    sep="\t",
    header=None,
    names=columns,
    dtype={
        "source_row_id": "int64",
        "split_name": "category",
        "amount_paid": "string",
        "payment_currency": "category",
        "prior_24h_count": "uint32",
        "is_laundering": "uint8",
    },
    parse_dates=["transaction_timestamp"],
)

expected_counts = {"train": 2_134_000, "validation": 965_524}
assert development["split_name"].value_counts().to_dict() == expected_counts
assert development["source_row_id"].is_unique
assert not development.isna().any().any()
assert pd.api.types.is_datetime64_any_dtype(development["transaction_timestamp"])

split_summary = development.groupby("split_name", observed=True).agg(
    transactions=("source_row_id", "size"),
    first_timestamp=("transaction_timestamp", "min"),
    last_timestamp=("transaction_timestamp", "max"),
    payment_currencies=("payment_currency", "nunique"),
).reindex(["train", "validation"])
assert split_summary["payment_currencies"].eq(15).all()
display(split_summary)

,transactions,first_timestamp,last_timestamp,payment_currencies
split_name,,,,
train,2134000,2022-09-02,2022-09-06 23:59:00,15
validation,965524,2022-09-07,2022-09-08 23:59:00,15


In [2]:
# Step 23. Audit repeated transaction records in the source CSV.
import numpy as np

source_path = Path("data/raw/HI-Small_Trans.csv")

with source_path.open("rb") as source:
    next(source)
    row_hashes = np.fromiter(
        (hash(row.rsplit(b",", 1)[0]) for row in source),
        dtype=np.int64,
    )

rows_checked = len(row_hashes)
unique_hashes, counts = np.unique(row_hashes, return_counts=True)
candidate_hashes = set(unique_hashes[counts > 1].tolist())
del row_hashes, unique_hashes, counts

seen = set()
repeated_rows = 0

with source_path.open("rb") as source:
    next(source)
    for row in source:
        fields = row.rsplit(b",", 1)[0]
        if hash(fields) in candidate_hashes:
            if fields in seen:
                repeated_rows += 1
            seen.add(fields)

print(f"Checked {rows_checked:,} source rows")
print(f"Found {repeated_rows:,} repeats beyond the first occurrence")

Checked 5,078,345 source rows
Found 9 repeats beyond the first occurrence


In [3]:
# Step 24. Summarise laundering labels by training date.
training_labels = development.loc[
    development["split_name"].eq("train"),
    ["transaction_timestamp", "is_laundering"],
].copy()

training_labels["transaction_date"] = (
    training_labels["transaction_timestamp"].dt.normalize()
)

training_by_day = training_labels.groupby("transaction_date").agg(
    transactions=("is_laundering", "size"),
    labelled_laundering=("is_laundering", "sum"),
)
training_by_day["label_rate_pct"] = (
    100 * training_by_day["labelled_laundering"] / training_by_day["transactions"]
)

display(training_by_day.round({"label_rate_pct": 3}))
print(f"Total labelled training rows: {training_by_day['labelled_laundering'].sum():,}")

,transactions,labelled_laundering,label_rate_pct
transaction_date,,,
2022-09-02,754449,408,0.054
2022-09-03,207382,391,0.189
2022-09-04,207430,407,0.196
2022-09-05,482650,471,0.098
2022-09-06,482089,531,0.110


Total labelled training rows: 2,208


In [4]:
# Step 25. Set the daily review budget for all three rankings.
review_budget_per_day = 500

training_capacity = training_by_day[["transactions"]].copy()
training_capacity["review_capacity_pct"] = (
    100 * review_budget_per_day / training_capacity["transactions"]
)

print(f"Daily review budget: {review_budget_per_day:,} transactions")
display(training_capacity.round({"review_capacity_pct": 3}))

Daily review budget: 500 transactions


,transactions,review_capacity_pct
transaction_date,,
2022-09-02,754449,0.066
2022-09-03,207382,0.241
2022-09-04,207430,0.241
2022-09-05,482650,0.104
2022-09-06,482089,0.104


In [5]:
# Step 26. Convert paid amounts to integer millionths.
amount_text = development["amount_paid"]
assert amount_text.str.fullmatch(r"[0-9]{1,13}\.[0-9]{6}").all()

development["amount_paid_millionths"] = (
    amount_text.str.replace(".", "", regex=False).astype("uint64")
)

display(
    development[["payment_currency", "amount_paid", "amount_paid_millionths"]].head()
)

,payment_currency,amount_paid,amount_paid_millionths
0,US Dollar,4115.090000,4115090000
1,Euro,3511.820000,3511820000
2,US Dollar,16135.090000,16135090000
3,US Dollar,12183.280000,12183280000
4,US Dollar,15197.450000,15197450000


In [6]:
# Step 27. Build the training references.
training_inputs = development.loc[
    development["split_name"].eq("train"),
    ["payment_currency", "amount_paid_millionths", "prior_24h_count"],
]

amount_reference = {
    currency: np.sort(rows["amount_paid_millionths"].to_numpy())
    for currency, rows in training_inputs.groupby("payment_currency", observed=True)
}
history_reference = np.sort(training_inputs["prior_24h_count"].to_numpy())

amount_rows = sum(len(amounts) for amounts in amount_reference.values())
print(f"Amount reference: {amount_rows:,} amounts across {len(amount_reference)} currencies")
print(f"History reference: {len(history_reference):,} counts")

Amount reference: 2,134,000 amounts across 15 currencies
History reference: 2,134,000 counts


In [7]:
# Step 28. Calculate both percentiles from the training references.
def reference_percentile(reference, values):
    below = np.searchsorted(reference, values, side="left")
    below_or_equal = np.searchsorted(reference, values, side="right")
    return (below + below_or_equal) / (2 * len(reference))


def add_percentile_features(rows, amount_reference, history_reference):
    amounts = rows["amount_paid_millionths"].to_numpy()
    amount_percentiles = np.full(len(rows), np.nan)

    for currency, reference in amount_reference.items():
        in_currency = rows["payment_currency"].eq(currency).to_numpy()
        amount_percentiles[in_currency] = reference_percentile(
            reference, amounts[in_currency]
        )

    if np.isnan(amount_percentiles).any():
        raise ValueError("Every scored currency needs a training reference.")

    rows["amount_percentile"] = amount_percentiles
    rows["history_percentile"] = reference_percentile(
        history_reference, rows["prior_24h_count"].to_numpy()
    )


add_percentile_features(development, amount_reference, history_reference)

display(development[[
    "payment_currency", "amount_paid", "amount_percentile",
    "prior_24h_count", "history_percentile",
]].head().round(6))

,payment_currency,amount_paid,amount_percentile,prior_24h_count,history_percentile
0,US Dollar,4115.090000,0.735512,0,0.072899
1,Euro,3511.820000,0.738019,0,0.072899
2,US Dollar,16135.090000,0.866136,2,0.299252
3,US Dollar,12183.280000,0.843430,3,0.397822
4,US Dollar,15197.450000,0.861379,4,0.484385


In [9]:
# Step 29. Evaluate the amount-only baseline on validation.
feature_columns = ["amount_percentile", "history_percentile"]
validation = development.loc[
    development["split_name"].eq("validation"),
    ["source_row_id", "transaction_timestamp", *feature_columns, "is_laundering"],
].copy()
validation["transaction_date"] = validation["transaction_timestamp"].dt.normalize()


def review_queue(rows, score_column, daily_budget):
    ranked = rows.sort_values(
        ["transaction_date", score_column, "transaction_timestamp", "source_row_id"],
        ascending=[True, False, True, True],
    )
    return ranked.groupby("transaction_date", sort=False).head(daily_budget)


def daily_review_summary(rows, queue):
    labelled = rows.groupby("transaction_date")["is_laundering"].sum()
    summary = queue.groupby("transaction_date").agg(
        reviewed=("source_row_id", "size"),
        caught=("is_laundering", "sum"),
    )
    summary["missed"] = labelled - summary["caught"]
    summary["false_positives"] = summary["reviewed"] - summary["caught"]
    return summary


amount_only_queue = review_queue(validation, "amount_percentile", review_budget_per_day)
amount_only_validation = daily_review_summary(validation, amount_only_queue)
display(amount_only_validation)

,reviewed,caught,missed,false_positives
transaction_date,,,,
2022-09-07,500,2,495,498
2022-09-08,500,1,538,499


In [10]:
# Step 30. Select a history weight on validation.
def history_ranking_score(rows, history_weight):
    return (
        (1 - history_weight) * rows["amount_percentile"]
        + history_weight * rows["history_percentile"]
    )


history_results = []
for weight in (0.25, 0.50, 0.75):
    scored = validation.assign(
        history_score=history_ranking_score(validation, weight)
    )
    queue = review_queue(scored, "history_score", review_budget_per_day)
    totals = daily_review_summary(scored, queue).sum()
    history_results.append({"history_weight": weight, **totals.to_dict()})

history_validation = (
    pd.DataFrame(history_results)
    .sort_values(["caught", "history_weight"], ascending=[False, True])
    .reset_index(drop=True)
)
selected_history_weight = float(history_validation.at[0, "history_weight"])

display(history_validation)
print(f"Selected history weight: {selected_history_weight:.0%}")

,history_weight,reviewed,caught,missed,false_positives
0,0.25,1000.0,0.0,1036.0,1000.0
1,0.50,1000.0,0.0,1036.0,1000.0
2,0.75,1000.0,0.0,1036.0,1000.0


Selected history weight: 25%


In [11]:
# Step 31. Select the decision tree depth on validation.
from sklearn.tree import DecisionTreeClassifier

train_mask = development["split_name"].eq("train")
X_train = development.loc[train_mask, feature_columns]
y_train = development.loc[train_mask, "is_laundering"]
X_validation = validation[feature_columns]

trees_by_depth = {}
tree_results = []

for depth in (2, 3, 4):
    tree = DecisionTreeClassifier(
        criterion="gini",
        max_depth=depth,
        min_samples_leaf=1_000,
        class_weight=None,
        random_state=42,
    )
    tree.fit(X_train, y_train)
    trees_by_depth[depth] = tree

    positive_class = list(tree.classes_).index(1)
    scored = validation.assign(
        tree_score=tree.predict_proba(X_validation)[:, positive_class]
    )
    queue = review_queue(scored, "tree_score", review_budget_per_day)
    totals = daily_review_summary(scored, queue).sum().astype("int64")
    tree_results.append({"max_depth": depth, **totals.to_dict()})

tree_validation = (
    pd.DataFrame(tree_results)
    .sort_values(["caught", "max_depth"], ascending=[False, True])
    .reset_index(drop=True)
)
selected_tree_depth = int(tree_validation.at[0, "max_depth"])
selected_tree = trees_by_depth[selected_tree_depth]

display(tree_validation)
print(f"Selected tree depth: {selected_tree_depth}")

,max_depth,reviewed,caught,missed,false_positives
0,4,1000,58,978,942
1,2,1000,12,1024,988
2,3,1000,12,1024,988


Selected tree depth: 4


In [12]:
# Step 32. Save the ranking setup before accessing test labels.
import joblib
import sklearn

frozen_path = Path("data/model/aml_frozen_rankings.joblib")

if not frozen_path.exists():
    bundle = {
        "feature_columns": tuple(feature_columns),
        "amount_reference": amount_reference,
        "history_reference": history_reference,
        "history_weight": selected_history_weight,
        "tree": selected_tree,
        "review_budget_per_day": review_budget_per_day,
        "tie_break": ("transaction_timestamp", "source_row_id"),
        "windows": {
            "train": ("2022-09-02", "2022-09-07"),
            "validation": ("2022-09-07", "2022-09-09"),
            "test": ("2022-09-09", "2022-09-11"),
        },
        "scikit_learn_version": sklearn.__version__,
    }
    temporary_path = frozen_path.with_suffix(".tmp")
    joblib.dump(bundle, str(temporary_path), compress=3)
    temporary_path.rename(frozen_path)

frozen_rankings = joblib.load(str(frozen_path))
print(f"Ranking setup saved at {frozen_path}")
print(f"History weight: {frozen_rankings['history_weight']:.0%}")
print(f"Tree depth: {frozen_rankings['tree'].max_depth}")
print(f"Reviews per day: {frozen_rankings['review_budget_per_day']}")

Ranking setup saved at data\model\aml_frozen_rankings.joblib
History weight: 25%
Tree depth: 4
Reviews per day: 500


In [13]:
# Step 34. Load and verify the held-out test rows.
test_dtypes = development.dtypes.reindex(columns).drop("transaction_timestamp").to_dict()
test_dtypes["split_name"] = "category"

test = pd.read_csv(
    Path("data/model/aml_ranking_test.tsv"),
    sep="\t",
    header=None,
    names=columns,
    dtype=test_dtypes,
    parse_dates=["transaction_timestamp"],
)

test_start, test_end = map(pd.Timestamp, frozen_rankings["windows"]["test"])
assert len(test) == 862_792
assert test["split_name"].eq("test").all()
assert test["source_row_id"].is_unique
assert not test["source_row_id"].isin(development["source_row_id"]).any()
assert not test.isna().any().any()
assert test["transaction_timestamp"].between(test_start, test_end, inclusive="left").all()
assert set(test["payment_currency"]) == set(frozen_rankings["amount_reference"])

test["transaction_date"] = test["transaction_timestamp"].dt.normalize()
display(test.groupby("transaction_date").size().to_frame("transactions"))

,transactions
transaction_date,
2022-09-09,654467
2022-09-10,208325


In [14]:
# Step 35. Compare the three frozen rankings on the held-out test window.
frozen_rankings = joblib.load(str(frozen_path))
assert sklearn.__version__ == frozen_rankings["scikit_learn_version"]

amount_text = test["amount_paid"]
assert amount_text.str.fullmatch(r"[0-9]{1,13}\.[0-9]{6}").all()
test["amount_paid_millionths"] = (
    amount_text.str.replace(".", "", regex=False).astype("uint64")
)

add_percentile_features(
    test,
    frozen_rankings["amount_reference"],
    frozen_rankings["history_reference"],
)
test["history_score"] = history_ranking_score(
    test, frozen_rankings["history_weight"]
)

frozen_tree = frozen_rankings["tree"]
test_features = test[list(frozen_rankings["feature_columns"])]
positive_class = list(frozen_tree.classes_).index(1)
test["tree_score"] = frozen_tree.predict_proba(test_features)[:, positive_class]

score_columns = {
    "Amount only": "amount_percentile",
    "Amount + history": "history_score",
    "Decision tree": "tree_score",
}

daily_budget = frozen_rankings["review_budget_per_day"]
test_queues = {}
test_daily = {}

for ranking, score_column in score_columns.items():
    queue = review_queue(test, score_column, daily_budget)
    test_queues[ranking] = queue
    test_daily[ranking] = daily_review_summary(test, queue).astype("int64")

test_comparison = pd.DataFrame({
    ranking: daily.sum() for ranking, daily in test_daily.items()
}).T
test_comparison.index.name = "ranking"

labelled_test_total = int(test["is_laundering"].sum())
assert test_comparison["reviewed"].eq(1_000).all()
assert (test_comparison["caught"] + test_comparison["missed"]).eq(labelled_test_total).all()

display(test_comparison)
print(f"Transactions labelled as laundering in the test window: {labelled_test_total:,}")

,reviewed,caught,missed,false_positives
ranking,,,,
Amount only,1000,7,949,993
Amount + history,1000,1,955,999
Decision tree,1000,53,903,947


Transactions labelled as laundering in the test window: 956


In [15]:
# Step 36. Select two reviewed transactions for the case notes.
tree_queue = test_queues["Decision tree"].copy()
tree_queue["daily_review_rank"] = (
    tree_queue.groupby("transaction_date").cumcount() + 1
)

case_examples = (
    tree_queue.sort_values(
        ["tree_score", "transaction_timestamp", "source_row_id"],
        ascending=[False, True, True],
    )
    .groupby("is_laundering", sort=False)
    .head(1)
    .sort_values("is_laundering", ascending=False)
    .copy()
)
case_examples["case"] = case_examples["is_laundering"].map({
    1: "Labelled laundering",
    0: "False positive",
})

case_features = case_examples[list(frozen_rankings["feature_columns"])]
leaf_ids = frozen_tree.apply(case_features)
nodes = frozen_tree.tree_
case_examples["leaf_id"] = leaf_ids
case_examples["leaf_training_rows"] = nodes.n_node_samples[leaf_ids]

case_fields = [
    "source_row_id", "transaction_timestamp", "daily_review_rank",
    "amount_paid", "payment_currency", "prior_24h_count",
    "amount_percentile", "history_percentile", "tree_score",
    "leaf_id", "leaf_training_rows",
]
display(case_examples.set_index("case")[case_fields].T)

paths = frozen_tree.decision_path(case_features)
for position, case_name in enumerate(case_examples["case"]):
    print(case_name)
    for node_id in paths[position].indices:
        feature_index = nodes.feature[node_id]
        if feature_index >= 0:
            threshold = nodes.threshold[node_id]
            value = np.float32(case_features.iloc[position, feature_index])
            relation = "<=" if value <= threshold else ">"
            print(f"  {case_features.columns[feature_index]} {relation} {threshold:.4f}")

case,Labelled laundering,False positive
source_row_id,3411041,4217219
transaction_timestamp,2022-09-09 00:25:00,2022-09-09 00:01:00
daily_review_rank,31,1
amount_paid,53292.350000,18605.460000
payment_currency,Saudi Riyal,US Dollar
prior_24h_count,0,0
amount_percentile,0.851031,0.876741
history_percentile,0.072899,0.072899
tree_score,0.024652,0.024652
leaf_id,12,12


Labelled laundering
  history_percentile <= 0.1351
  amount_percentile > 0.5685
  amount_percentile <= 0.8815
  amount_percentile > 0.8286
False positive
  history_percentile <= 0.1351
  amount_percentile > 0.5685
  amount_percentile <= 0.8815
  amount_percentile > 0.8286


In [16]:
# Step 37. Export the held-out comparison for Power BI.
power_bi_results = pd.concat(
    test_daily, names=["ranking", "transaction_date"]
).reset_index()

assert len(power_bi_results) == 6
assert power_bi_results["reviewed"].eq(daily_budget).all()

report_totals = power_bi_results.groupby("ranking")[
    test_comparison.columns.tolist()
].sum()
pd.testing.assert_frame_equal(
    report_totals.reindex(test_comparison.index), test_comparison
)

report_path = Path("outputs/aml_test_ranking_results.csv")
report_path.parent.mkdir(exist_ok=True)
power_bi_results.to_csv(report_path, index=False, date_format="%Y-%m-%d")

display(power_bi_results)
print(f"Saved {len(power_bi_results)} rows to {report_path}")

,ranking,transaction_date,reviewed,caught,missed,false_positives
0,Amount only,2022-09-09,500,1,513,499
1,Amount only,2022-09-10,500,6,436,494
2,Amount + history,2022-09-09,500,1,513,499
3,Amount + history,2022-09-10,500,0,442,500
4,Decision tree,2022-09-09,500,4,510,496
5,Decision tree,2022-09-10,500,49,393,451


Saved 6 rows to outputs\aml_test_ranking_results.csv
